# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [30]:

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("DuckDB ready, HF secret registered.")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"
print(f"Working month: {MONTH}")

DuckDB ready, HF secret registered.
Working month: 2026-03


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*



Building on the data contract from ML-04, this pulls one row per page (collapsing the month's
daily rows, same fix needed in ML-04's leakage trap), with engineered features, categorical
handling, and explicit fill logic.

In [31]:
raw = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id, f.report_date,
        f.gsc_clicks, f.gsc_impressions, f.gsc_avg_position,
        f.ga4_data_available,
        c.content_type, c.word_count, c.main_intent, c.competition_level,
        c.content_created_date, c.is_deleted, c.is_published
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
    JOIN read_parquet('{BASE}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
    WHERE c.is_deleted = FALSE AND c.is_published = TRUE
""").df()
print(f"{len(raw):,} raw page-day rows pulled")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

9,532,718 raw page-day rows pulled


In [32]:
agg = raw.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
    content_type=("content_type", "first"),
    word_count=("word_count", "first"),
    main_intent=("main_intent", "first"),
    competition_level=("competition_level", "first"),
    content_created_date=("content_created_date", "first"),
    days_seen=("report_date", "nunique"),
).reset_index()

agg["ctr"] = (agg["gsc_clicks"] / agg["gsc_impressions"].replace(0, np.nan)).fillna(0).round(4)
agg["content_age_days"] = (pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(agg["content_created_date"])).dt.days
agg["log_impressions"] = np.log1p(agg["gsc_impressions"])
agg["position_tier"] = pd.cut(agg["gsc_avg_position"],
                                bins=[0, 3, 10, 20, 1000],
                                labels=["top_3", "page_1", "page_2_3", "deep"])

for cat_col in ["content_type", "main_intent", "competition_level"]:
    agg[f"{cat_col}_missing"] = agg[cat_col].isna().astype(int)
    agg[cat_col] = agg[cat_col].fillna("unknown")

print(f"{len(agg):,} unique pages in the feature vector")
agg[["content_hash_id", "ctr", "gsc_avg_position", "position_tier", "content_type",
     "word_count", "content_age_days", "log_impressions"]].head(10)

321,106 unique pages in the feature vector


,content_hash_id,ctr,gsc_avg_position,position_tier,content_type,word_count,content_age_days,log_impressions
0,content_000005d4ced12088,0.0,72.854861,deep,keyword article,<NA>,338,4.465908
1,content_00001e488b74b799,0.0,NaN,NaN,keyword article,<NA>,317,0.000000
2,content_00007bd2985b77c3,0.0,5.269565,page_1,keyword article,<NA>,213,3.871201
3,content_00008950670cb6b5,0.0,NaN,NaN,keyword article,2005,233,0.000000
4,content_0000a348850eb1fc,0.0,NaN,NaN,feedly article,811,180,0.000000
5,content_0000c57e204651e5,0.0,NaN,NaN,feedly article,836,209,0.000000
6,content_0000cd28fbda69f3,0.0,4.251282,page_1,feedly article,990,187,3.401197
7,content_0000d31f3926ea12,0.0,NaN,NaN,feedly article,868,195,0.000000
8,content_0000d495bfbfb4a8,0.0,3.333333,page_1,keyword article,2832,12,2.772589
9,content_0000feb69f0f60db,0.0,NaN,NaN,keyword article,1102,169,0.000000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*



| Feature | Meaning | Missing handling | Available before decision moment? |
|---|---|---|---|
| `ctr` | Clicks / impressions for the page across the month | Filled 0 if impressions=0 | Yes — purely observed, current-window data |
| `gsc_avg_position` | Mean search ranking across the month | Rows require impressions>0 already | Yes — observed daily |
| `position_tier` | Bucketed version of `gsc_avg_position` | Derived, no separate missingness | Yes — pure derivation of the above |
| `content_type` | Page format category | Filled "unknown" + flagged with `content_type_missing` | Yes — static metadata, set at creation |
| `word_count` | Page length | No fill needed — comes straight from `dim_content` | Yes — static content property |
| `content_age_days` | Days since content creation, relative to this month | No fill needed — computed from a date | Yes — a pure calendar fact |
| `log_impressions` | Log-scaled impression volume | No fill needed — derived from `ctr`'s inputs | Yes — same observed signal as CTR, just rescaled |
| `main_intent` | Search intent category | Filled "unknown" + flagged | Yes — static metadata |
| `competition_level` | Keyword competitiveness bucket | Filled "unknown" + flagged | Yes — static metadata |

Missingness pattern check — is it random, or does it follow content_type (as the data skill
warns it usually does)?

In [33]:
missing_by_type = raw.groupby("content_type")[["main_intent", "competition_level"]].apply(
    lambda g: g.isna().mean()
).round(3)
print("Missing rate by content_type (0 = never missing, 1 = always missing):")
missing_by_type

Missing rate by content_type (0 = never missing, 1 = always missing):


,main_intent,competition_level
content_type,,
comparison article,0.001,0.001
feedly article,1.000,1.000
keyword article,0.025,0.033


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*



Attacking my own feature vector on purpose, checking for the three classic traps: a
label-derived column, a future-window column, and a product-decision flag sneaking in — using
the full checklist from `hunting-leakage-and-validating` (base rate, grouped split,
with/without test, feature importance).

In [34]:
# Step 1: define the label on the full (unfiltered) dataset first
agg["label_low_ctr"] = (agg["ctr"] < agg["ctr"].median()).astype(int)
print(f"Label balance on unfiltered agg:\n{agg['label_low_ctr'].value_counts(normalize=True).round(4)}")

Label balance on unfiltered agg:
label_low_ctr
0    1.0
Name: proportion, dtype: float64


In [35]:
# Diagnose: is the label degenerate because the median CTR is 0?
print(f"Median CTR: {agg['ctr'].median()}")
print(f"CTR value counts (top 5):\n{agg['ctr'].value_counts().head(5)}")
print(f"\nLabel balance: {agg['label_low_ctr'].value_counts(normalize=True).round(4)}")

Median CTR: 0.0
CTR value counts (top 5):
ctr
0.0000    252312
0.0013      1396
0.0014      1396
0.0009      1385
0.0012      1384
Name: count, dtype: int64

Label balance: label_low_ctr
0    1.0
Name: proportion, dtype: float64


In [36]:
# Refilter to pages with meaningful volume before defining the label
agg_filtered = agg[agg["gsc_impressions"] >= 20].copy()
print(f"{len(agg_filtered):,} pages after volume filter (was {len(agg):,})")

agg_filtered["label_low_ctr"] = (agg_filtered["ctr"] < agg_filtered["ctr"].median()).astype(int)
print(f"New median CTR: {agg_filtered['ctr'].median()}")
print(f"New label balance:\n{agg_filtered['label_low_ctr'].value_counts(normalize=True).round(4)}")

132,471 pages after volume filter (was 321,106)
New median CTR: 0.0003
New label balance:
label_low_ctr
0    0.5011
1    0.4989
Name: proportion, dtype: float64


In [37]:
# ATTACK 1 (per hunting-leakage-and-validating checklist): base rate, with/without test,
# grouped split, feature importance — run on the FIXED label
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.dummy import DummyClassifier

safe_features = ["word_count", "content_age_days", "content_type_missing"]
X = pd.get_dummies(agg_filtered[safe_features + ["content_type"]], columns=["content_type"])
y = agg_filtered["label_low_ctr"]
groups = agg_filtered["client_hash_id"]

base_rate = y.mean()
print(f"Base rate (majority class): {max(base_rate, 1-base_rate):.3f}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
tree_random = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(f"Random split score: {tree_random.score(X_test, y_test):.3f}")

gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))
tree_grouped = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
grouped_score = tree_grouped.score(X.iloc[test_idx], y.iloc[test_idx])
print(f"Grouped (by-client) split score: {grouped_score:.3f}")
print(f"Gap between random and grouped: {tree_random.score(X_test, y_test) - grouped_score:.3f}")

X_without_type = X[[c for c in X.columns if not c.startswith("content_type_")]]
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_without_type, y, test_size=0.3, random_state=42, stratify=y)
tree_without = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xw_train, yw_train)
print(f"\nScore WITHOUT content_type: {tree_without.score(Xw_test, yw_test):.3f}")
print(f"Score WITH content_type:    {tree_random.score(X_test, y_test):.3f}")

importances = pd.Series(tree_random.feature_importances_, index=X.columns).sort_values(ascending=False)
print(f"\nTop features by importance:\n{importances.head(5)}")

Base rate (majority class): 0.501
Random split score: 0.654
Grouped (by-client) split score: 0.657
Gap between random and grouped: -0.003

Score WITHOUT content_type: 0.648
Score WITH content_type:    0.654

Top features by importance:
word_count                         0.633386
content_age_days                   0.242092
content_type_keyword article       0.124522
content_type_missing               0.000000
content_type_comparison article    0.000000
dtype: float64


**What the checklist found:** defining the label on the full, unfiltered dataset scored 1.000
— but investigating it (rather than celebrating it, per the skill's own instruction) revealed
the real cause wasn't leakage: the label was degenerate. 78.6% of pages had zero CTR (252,312
of 321,106), pushing the median CTR to exactly 0.0, which made `ctr < median` almost always
false — a broken label, not a broken model.

**The fix:** applying a `gsc_impressions >= 20` volume filter dropped the dataset from 321,106
to 132,471 pages, but produced a real, balanced label (50.1% / 49.9%) with a sensible median
CTR (0.0003).

**Results on the fixed label, run through the full checklist:**
- Base rate: 0.501 — a coin flip, as expected from a balanced label
- Random split: 0.654 vs. grouped-by-client split: 0.657 — a gap of only -0.003, meaning the
  model isn't secretly memorizing client-level patterns; a clean, honest split
- With vs. without `content_type`: 0.654 vs. 0.648 — barely any difference, so content_type
  isn't quietly doing all the work
- Top feature by importance: `word_count` (0.63), followed by `content_age_days` (0.24) — a
  believable, non-suspicious result

**Lesson:** a perfect score is not proof of leakage — it's a prompt to investigate. Here the
real bug was upstream, in how the label itself was defined, not in the features. The base-rate
and with/without checks from the `hunting-leakage-and-validating` skill caught it; a bare
accuracy number alone would have hidden the problem.

**Note going forward:** `agg_filtered` (132,471 pages) is the working, honest dataset from
here on — `agg` (321,106 pages, unfiltered) is kept only as evidence for this investigation.

In [38]:
# ATTACK 2: does my feature window overlap the label's own time window?
print("Feature window: 2026-03-01 to 2026-03-31 (content_age_days computed relative to this)")
print("Label window:   same month (2026-03), since ctr and label_low_ctr both come from it")
print("VERDICT: same-window scoring is fine for a static opportunity ranking, but would be")
print("leakage if reused as a 'predict future decline' label without a forward time split.")

Feature window: 2026-03-01 to 2026-03-31 (content_age_days computed relative to this)
Label window:   same month (2026-03), since ctr and label_low_ctr both come from it
VERDICT: same-window scoring is fine for a static opportunity ranking, but would be
leakage if reused as a 'predict future decline' label without a forward time split.


In [39]:
# ATTACK 3: did any product-decision column sneak into the raw pull?
product_flag_names = ["health_score", "priority_score", "action_type", "needs_ctr_fix",
                        "is_quick_win", "refresh_tier"]
leaked = [c for c in product_flag_names if c in agg_filtered.columns]
print(f"Product-decision columns found in my feature vector: {leaked}")
print("Empty list = clean. These aren't shipped in this dataset per the data skill, confirmed here.")

Product-decision columns found in my feature vector: []
Empty list = clean. These aren't shipped in this dataset per the data skill, confirmed here.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*



- **`trend_direction`, `trend_pct`** — not present in the warehouse fact table at all (unlike
  the starter CSV) — confirmed the hard way in ML-04 when a query referencing `trend_pct`
  errored out. Even if they existed, they'd be leakage traps derived from the same outcome.
- **`health_score`, `priority_score`, `action_type`** — product-decision outputs, not shipped
  in this dataset; confirmed absent in the Attack 3 check above. Would be circular if
  reconstructed and reused as features.
- **`keyword_hash_id`, `url_hash_id`, `client_hash_id`, `content_hash_id`** — pseudonymized
  IDs, used only for joining/grouping/deduplication, never as model inputs.
- **Rows before a client's `ga4_data_start`** — GA4 fields are zero-filled there, not truly
  zero, so any engagement-based feature must filter on `ga4_data_available` first.
- **`is_deleted`, `is_published`** — used only as a filter to keep the dataset clean, never as
  model features, since they reflect moderation state rather than a search signal.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.